In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
# 1. Load the original dataset (specifying ';' as the separator)
df = pd.read_csv('out/ta75_results.csv', sep=';')

# 2. Identify the parameter columns to group by
param_columns = ['InitialTemp', 'FinalTemp', 'MaxIters', 'Version', 'NeighSize']

# 3. Group by parameters and calculate the mean for Cmax and Duration_ms
aggregated_df = df.groupby(param_columns)[['Cmax', 'Duration_ms']].mean().reset_index()

# 4. Rename the columns to reflect that they are mean values
aggregated_df = aggregated_df.rename(columns={
    'Cmax': 'Mean_Cmax',
    'Duration_ms': 'Mean_Duration_ms'
})

# 5. Export the aggregated data to a new CSV file
aggregated_df.to_csv('out/ta75_results_aggregated.csv', sep=';', index=False)

In [27]:
# 1. Load the aggregated dataset
df_agg = pd.read_csv('out/ta75_results_aggregated.csv', sep=';')

# --- FIND BEST CMAX PER VERSION ---
# Sort by Mean_Cmax ascending, and use Mean_Duration_ms to break ties 
df_sorted_cmax = df_agg.sort_values(by=['Mean_Cmax', 'Mean_Duration_ms'])
# Group by Version and select the row index with the minimum Mean_Cmax
best_cmax_per_version = df_sorted_cmax.loc[df_sorted_cmax.groupby('Version')['Mean_Cmax'].idxmin()]

# --- FIND BEST DURATION PER VERSION ---
# Group by Version and select the row index with the minimum Mean_Duration_ms
best_duration_per_version = df_agg.loc[df_agg.groupby('Version')['Mean_Duration_ms'].idxmin()]

# 3. Display results
print("=== BEST CMAX CONFIGURATIONS ===")
print(best_cmax_per_version[['Version', 'InitialTemp', 'FinalTemp', 'MaxIters', 'NeighSize', 'Mean_Cmax', 'Mean_Duration_ms']])

print("\n=== BEST DURATION CONFIGURATIONS ===")
print(best_duration_per_version[['Version', 'InitialTemp', 'FinalTemp', 'MaxIters', 'NeighSize', 'Mean_Cmax', 'Mean_Duration_ms']])

=== BEST CMAX CONFIGURATIONS ===
   Version  InitialTemp  FinalTemp  MaxIters  NeighSize  Mean_Cmax  \
33    BOTH       1000.0      0.001    400000        200     8314.9   
37  GREEDY       1000.0      0.001    400000        200     8306.0   
38  NORMAL       1000.0      0.001    400000          1     8659.5   
39  REHEAT       1000.0      0.001    400000          1     8690.3   

    Mean_Duration_ms  
33           83785.9  
37           83631.2  
38             437.2  
39             433.6  

=== BEST DURATION CONFIGURATIONS ===
   Version  InitialTemp  FinalTemp  MaxIters  NeighSize  Mean_Cmax  \
0     BOTH       1000.0      0.001     20000         50     8528.2   
84  GREEDY     100000.0      0.001     20000         50     8584.9   
48  NORMAL      10000.0      0.001     20000          1     9084.2   
9   REHEAT       1000.0      0.001     20000          1     8990.7   

    Mean_Duration_ms  
0             1103.6  
84            1109.4  
48              27.4  
9               27.2

In [28]:
# ta05
# best_params = {
#     'BOTH':   {'InitialTemp': 1000000.0, 'FinalTemp': 0.001, 'MaxIters': 2250,  'NeighSize': 22},
#     'GREEDY': {'InitialTemp': 1000000.0, 'FinalTemp': 0.001, 'MaxIters': 11250, 'NeighSize': 15},
#     'NORMAL': {'InitialTemp': 1000.0,    'FinalTemp': 0.001, 'MaxIters': 45000, 'NeighSize': 1},
#     'REHEAT': {'InitialTemp': 1000.0,    'FinalTemp': 0.001, 'MaxIters': 45000, 'NeighSize': 1}
# }

# ta45
# best_params = {
#     'BOTH':   {'InitialTemp': 1000000.0, 'FinalTemp': 0.001, 'MaxIters': 120000,  'NeighSize': 60},
#     'GREEDY': {'InitialTemp': 1000000.0, 'FinalTemp': 0.001, 'MaxIters': 120000, 'NeighSize': 60},
#     'NORMAL': {'InitialTemp': 10000.0,    'FinalTemp': 0.001, 'MaxIters': 120000, 'NeighSize': 1},
#     'REHEAT': {'InitialTemp': 1000.0,    'FinalTemp': 0.001, 'MaxIters': 120000, 'NeighSize': 1}
# }

# ta75
best_params = {
    'BOTH':   {'InitialTemp': 1000.0, 'FinalTemp': 0.001, 'MaxIters': 400000,  'NeighSize': 200},
    'GREEDY': {'InitialTemp': 1000.0, 'FinalTemp': 0.001, 'MaxIters': 400000, 'NeighSize': 200},
    'NORMAL': {'InitialTemp': 1000.0,    'FinalTemp': 0.001, 'MaxIters': 400000, 'NeighSize': 1},
    'REHEAT': {'InitialTemp': 1000.0,    'FinalTemp': 0.001, 'MaxIters': 400000, 'NeighSize': 1}
}

# Custom color scheme for consistency
colors = {'BOTH': '#0052cc', 'GREEDY': '#e68a00', 'NORMAL': '#00a372', 'REHEAT': '#cc5200'}

In [29]:
# --- PLOT 1: InitialTemp vs Cmax (All 4 Versions) ---
fig, ax = plt.subplots(figsize=(8, 5))
for version in df_agg['Version'].unique():
    p = best_params[version]
    # Filter to isolate only this version and keep the OTHER 3 parameters fixed
    df_filtered = df_agg[
        (df_agg['Version'] == version) &
        (df_agg['FinalTemp'] == p['FinalTemp']) &
        (df_agg['MaxIters'] == p['MaxIters']) &
        (df_agg['NeighSize'] == p['NeighSize'])
    ].sort_values('InitialTemp')
    
    ax.plot(df_filtered['InitialTemp'], df_filtered['Mean_Cmax'], 
            marker='o', label=version, color=colors[version])

ax.set_xscale('log')
ax.set_xlabel('Initial Temperature (Log Scale)')
ax.set_ylabel('Mean $C_{max}$')
ax.set_title('Initial Temperature vs $C_{max}$ (Other Parameters Constant)')
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig('plots/ta75_initial_temp_vs_cmax.png')
plt.close()

In [30]:
# --- PLOT 2: Max Iters vs Cmax (All 4 Versions) ---
fig, ax = plt.subplots(figsize=(8, 5))
for version in df_agg['Version'].unique():
    p = best_params[version]
    df_filtered = df_agg[
        (df_agg['Version'] == version) &
        (df_agg['InitialTemp'] == p['InitialTemp']) &
        (df_agg['FinalTemp'] == p['FinalTemp']) &
        (df_agg['NeighSize'] == p['NeighSize'])
    ].sort_values('MaxIters')
    
    ax.plot(df_filtered['MaxIters'], df_filtered['Mean_Cmax'], 
            marker='o', label=version, color=colors[version])

ax.set_xlabel('Maximum Iterations')
ax.set_ylabel('Mean $C_{max}$')
ax.set_title('Max Iterations vs $C_{max}$ (Other Parameters Constant)')
ax.legend()
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig('plots/ta75_max_iters_vs_cmax.png')
plt.close()

In [31]:
# --- PLOT 3: Neighborhood Size vs Cmax (BOTH & GREEDY only) ---
fig, ax = plt.subplots(figsize=(8, 5))
for version in ['BOTH', 'GREEDY']:
    p = best_params[version]
    df_filtered = df_agg[
        (df_agg['Version'] == version) &
        (df_agg['InitialTemp'] == p['InitialTemp']) &
        (df_agg['FinalTemp'] == p['FinalTemp']) &
        (df_agg['MaxIters'] == p['MaxIters'])
    ].sort_values('NeighSize')
    
    ax.plot(df_filtered['NeighSize'], df_filtered['Mean_Cmax'], 
            marker='o', label=version, color=colors[version])

ax.set_xlabel('Neighborhood Size')
ax.set_ylabel('Mean $C_{max}$')
ax.set_title('Neighborhood Size vs $C_{max}$ (Other Parameters Constant)')
ax.legend()
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig('plots/ta75_neigh_size_vs_cmax.png')
plt.close()

In [32]:
# --- PLOT 4: MaxIters vs time (all versions) ---
fig, ax = plt.subplots(figsize=(8, 5))

for version in df_agg['Version'].unique():
    p = best_params[version]
    
    # Filter the dataframe: hold InitialTemp, FinalTemp, and NeighSize constant
    df_filtered = df_agg[
        (df_agg['Version'] == version) &
        (df_agg['InitialTemp'] == p['InitialTemp']) &
        (df_agg['FinalTemp'] == p['FinalTemp']) &
        (df_agg['NeighSize'] == p['NeighSize'])
    ].sort_values('MaxIters')
    
    # Plot MaxIters against Mean_Duration_ms instead of Mean_Cmax
    ax.plot(df_filtered['MaxIters'], df_filtered['Mean_Duration_ms'], 
            marker='o', label=version, color=colors[version])

# Formatting the plot axes and labels
ax.set_xlabel('Maximum Iterations')
ax.set_ylabel('Mean Duration (ms)')
ax.set_title('Max Iterations vs Duration (Other Parameters Constant)')
ax.legend()
ax.grid(True, ls="--", alpha=0.5)

# Optimize placement and save the figure
plt.tight_layout()
plt.savefig('plots/ta75_max_iters_constant_vs_duration.png')
plt.close()

In [49]:
# 1. Find all files that match the pattern 'taXX_results.csv'
# This matches files where XX is any combination of digits/characters
files = sorted(glob.glob("out/ta*_results_aggregated.csv"))

# List to hold the extracted row data for each file
summary_data = []

# 2. Iterate through each discovered file
for file_path in files:
    # Read the file (using ';' as the delimiter)
    df = pd.read_csv(file_path, sep=';')
    
    # Initialize the row dictionary with the file name
    file_name = os.path.basename(file_path)
    row_dict = {'filename': file_name}
    
    # 3. For each algorithm variant, find the best performance row
    for version in ['NORMAL', 'GREEDY', 'REHEAT', 'BOTH']:
        df_version = df[df['Version'] == version]
        
        if not df_version.empty:
            # Sort by Cmax ascending to get the minimum value. 
            # If there's a tie for Cmax, sort by Duration_ms ascending to find the quickest one.
            best_row = df_version.sort_values(by=['Mean_Cmax', 'Mean_Duration_ms']).iloc[0]
            
            row_dict[f'{version}_cmax'] = best_row['Mean_Cmax']
            row_dict[f'{version}_duration'] = best_row['Mean_Duration_ms']
        else:
            # Handle cases where a specific variant might be missing in a file
            row_dict[f'{version}_cmax'] = None
            row_dict[f'{version}_duration'] = None
            
    summary_data.append(row_dict)

# 4. Convert the gathered rows into a unified DataFrame
df_summary = pd.DataFrame(summary_data)

# 5. Ensure the columns follow your exact specified header and casing
df_summary = df_summary[[
    'filename',
    'NORMAL_cmax', 'NORMAL_duration',
    'GREEDY_cmax', 'GREEDY_duration',
    'REHEAT_cmax', 'REHEAT_duration',
    'BOTH_cmax', 'BOTH_duration'
]]

# 6. Save the compiled results to a new summary CSV file
df_summary.to_csv('out/all_files_best_summary.csv', sep=';', index=False)

print(f"Summary generated successfully for {len(files)} files!")

Summary generated successfully for 16 files!
